# 02 · Exploratory Data Analysis

**Input:** `../data/processed/master_cleaned_2015_2025.csv` (built by `01_data_cleaning.ipynb`)
**Output:** charts in `../outputs/eda_<date>/figures/` and one markdown explanation per chart.

Every figure below is paired with a written interpretation in
`CHART_EXPLANATIONS.md` **and** an individual `.md` next to the image, so a reader who
opens the folder without running anything still knows what they are looking at and — more
importantly — what the chart does **not** prove.

> These are descriptive statistics over a harmonised panel. The target in the ML notebooks
> is a *synthesized* low-NDVI label, not measured crop loss. Nothing here establishes
> causation, forecast skill, or forage availability.


In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")          # headless: write files, never open a window
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROC_DIR = Path("../data/processed")
RUN_DIR = Path("../outputs") / f"eda_{pd.Timestamp.today():%Y_%m_%d}"
FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("default")
plt.rcParams.update({"figure.dpi": 140, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False})

PANEL_PATH = PROC_DIR / "master_cleaned_2015_2025.csv"
if not PANEL_PATH.exists():
    raise FileNotFoundError(
        f"{PANEL_PATH} not found. Run 01_data_cleaning.ipynb first — it builds "
        "master_cleaned_2015_2025.csv (6,204 rows = 47 counties x 132 months) from the "
        "raw files downloaded by 00a_raw_data_acquisition.ipynb."
    )
PANEL = pd.read_csv(PANEL_PATH)
PANEL["date"] = pd.to_datetime(dict(year=PANEL.year, month=PANEL.month, day=1))

EXPLANATIONS = {}      # stem -> markdown text, written out at the end


def save(fig, stem, title, explanation):
    """Save a figure and register its explanation. Every chart goes through here,
    so a figure can never be written without the text that explains it."""
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{stem}.png", bbox_inches="tight")
    plt.close(fig)
    EXPLANATIONS[stem] = f"# {title}\n\n![{title}](figures/{stem}.png)\n\n{explanation}\n"
    print(f"  {stem}.png")


print(f"Panel: {len(PANEL):,} rows | {PANEL.county.nunique()} counties | "
      f"{PANEL.date.min():%Y-%m} to {PANEL.date.max():%Y-%m}")
print(f"Output: {RUN_DIR}")
print(f"Panel SHA-256: {hashlib.sha256((PROC_DIR / 'master_cleaned_2015_2025.csv').read_bytes()).hexdigest()[:16]}...")



## 1 · Coverage — is the panel actually complete?

Before any statistic, confirm the panel is 47 counties × 132 months with no holes. Every
later chart inherits any gap here, so this comes first.


In [ ]:
counts = PANEL.groupby("county").size().sort_values()
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(range(len(counts)), counts.values, color="#2b6842")
ax.axhline(132, color="#b23b3b", ls="--", lw=1, label="Expected 132 months")
ax.set(xticks=[], xlabel="47 counties (sorted)", ylabel="Months present",
       title="Coverage: months observed per county")
ax.legend(frameon=False)
save(fig, "01_coverage_by_county", "Coverage: months observed per county",
f"""Each bar is one county; the dashed line is the 132 months in 2015-01 to 2025-12.

**Reading it:** all 47 bars reach the line, so the panel is balanced — {len(PANEL):,} rows
= 47 x 132.

**Why it matters:** an unbalanced panel would break the lag and rolling-mean features in
`ml-models/notebooks/`, because a missing month silently shifts every subsequent lag. The
validation cell in `01_data_cleaning.ipynb` fails the build if a county is short.

**What it does not show:** that the *values* are complete or correct. Coverage is about
rows, not quality.""")


## 2 · Seasonality — rainfall and vegetation by calendar month


In [ ]:
monthly = PANEL.groupby("month")[["rainfall_mm", "ndvi_mean", "temp_mean_c", "soil_moisture"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].bar(monthly.index, monthly.rainfall_mm, color="#5b8fb9")
axes[0].set(xlabel="Calendar month", ylabel="Mean rainfall (mm)", title="Rainfall seasonality",
            xticks=range(1, 13))
axes[1].plot(monthly.index, monthly.ndvi_mean, marker="o", color="#2b6842")
axes[1].set(xlabel="Calendar month", ylabel="Mean NDVI", title="Vegetation seasonality",
            xticks=range(1, 13))
save(fig, "02_seasonality_rainfall_ndvi", "Seasonality: rainfall and vegetation by month",
f"""Left: mean rainfall per calendar month across all counties and years. Right: mean NDVI.

**Reading it:** rainfall peaks in the long rains (March–May) and again in the short rains
(October–December), with a dry January–February. NDVI tracks it with a lag: greenness rises
*after* the rains, not with them. Peak rainfall month {int(monthly.rainfall_mm.idxmax())}
vs peak greenness month {int(monthly.ndvi_mean.idxmax())}.

**Why it matters:** this lag is the physical reason the model's NDVI-lag and rolling
features carry signal. It is also why same-month weather cannot be an early warning — the
rain that drives greenness arrives before the greenness does.

**What it does not show:** causation, or that the relationship is stable per county. The
next charts disaggregate.""")


## 3 · The 2015–2025 trend


In [ ]:
annual = PANEL.groupby("year")[["ndvi_mean", "rainfall_mm", "temp_mean_c"]].mean()

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(annual.index, annual.ndvi_mean, marker="o", color="#2b6842", label="Mean NDVI")
ax.set(xlabel="Year", ylabel="Mean NDVI", title="Annual mean vegetation greenness, 2015–2025")
ax2 = ax.twinx()
ax2.plot(annual.index, annual.rainfall_mm, marker="s", color="#5b8fb9", alpha=.7, label="Mean rainfall")
ax2.set_ylabel("Mean rainfall (mm)")
ax.grid(False)
lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [l.get_label() for l in lines], frameon=False, loc="lower left")
save(fig, "03_annual_ndvi_rainfall", "Annual mean NDVI and rainfall, 2015–2025",
f"""Two axes: NDVI on the left, rainfall on the right. Both are unweighted means over counties.

**Reading it:** the NDVI range across eleven years is narrow
({annual.ndvi_mean.min():.3f}–{annual.ndvi_mean.max():.3f}), so year-on-year greenness barely
moves at the national aggregate. The lowest year is {int(annual.ndvi_mean.idxmin())} and the
highest {int(annual.ndvi_mean.idxmax())}.

**Why it matters:** a flat national mean means the interesting variation is *county-level
and seasonal*, not national-annual. Any model that only used a national time trend would
have almost nothing to learn from.

**What it does not show:** a trend over time in either direction. Eleven points cannot
support a trend claim, and the last years of such a series are typically still subject to
satellite-product revisions.""")


## 4 · Which counties are consistently greenest and driest?


In [ ]:
ranks = PANEL.groupby("county").ndvi_mean.mean().sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#b23b3b" if v < ranks.quantile(.25) else "#2b6842" if v > ranks.quantile(.75) else "#9a9a9a"
          for v in ranks.values]
ax.barh(range(len(ranks)), ranks.values, color=colors)
ax.set(yticks=range(len(ranks)), yticklabels=ranks.index, xlabel="Mean NDVI (2015–2025)",
       title="Counties ranked by long-run mean NDVI")
ax.tick_params(axis="y", labelsize=7)
save(fig, "04_county_ndvi_ranking", "Counties ranked by long-run mean NDVI",
f"""Bars are 11-year means per county. Red = bottom quartile, green = top quartile.

**Reading it:** the spread is wide — {ranks.min():.3f} ({ranks.index[0]}) to
{ranks.max():.3f} ({ranks.index[-1]}). Arid northern counties (Turkana, Marsabit, Wajir,
Mandera, Garissa) sit at the bottom; highland and western counties at the top.

**Why it matters:** the model scores each county against *its own* historical distribution,
not a national threshold. A fixed national cut-off would label every northern county as
permanently at risk and never fire in the highlands. This chart is why the target is
county-relative.

**What it does not show:** that low NDVI means crop loss. Low average greenness is a
climate fact in the arid north, not an event.""")


## 5 · Distribution and outliers


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, col, unit in zip(axes, ["ndvi_mean", "rainfall_mm", "soil_moisture"],
                         ["NDVI", "mm", "0–1 fraction"]):
    ax.hist(PANEL[col].dropna(), bins=50, color="#2b6842", alpha=.8)
    ax.set(xlabel=f"{col} ({unit})", ylabel="County-months", title=col)
save(fig, "05_distributions", "Distributions of the core variables",
f"""Histograms of every county-month observation.

**Reading it:** NDVI is roughly bell-shaped around {PANEL.ndvi_mean.mean():.2f}. Rainfall is
strongly right-skewed — most county-months are fairly dry, a tail runs to
{PANEL.rainfall_mm.max():.0f} mm. Soil moisture is left-skewed and clustered low.

**Why it matters:** the skew in rainfall is why the pipeline uses rainfall *anomalies* and
3-month totals rather than raw millimetres — a linear model on raw skewed rainfall would be
dominated by a handful of wet months.

**What it does not show:** whether outliers are errors. Extreme rainfall here is plausible
for Kenya; the checks in notebook 01 reject only physically impossible values.""")


## 6 · Correlation structure


In [ ]:
cols = ["ndvi_mean", "rainfall_mm", "temp_mean_c", "soil_moisture", "farming", "irrigation"]
corr = PANEL[cols].corr()

fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set(xticks=range(len(cols)), yticks=range(len(cols)), yticklabels=cols)
ax.set_xticklabels(cols, rotation=45, ha="right")
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.iloc[i, j]) > .6 else "black")
fig.colorbar(im, ax=ax, shrink=.8)
save(fig, "06_correlation_matrix", "Correlation matrix of core variables",
f"""Pearson correlation across all county-months.

**Reading it:** NDVI correlates with soil moisture (r = {corr.loc['ndvi_mean','soil_moisture']:.2f})
more than with same-month rainfall (r = {corr.loc['ndvi_mean','rainfall_mm']:.2f}). Temperature
correlates negatively (r = {corr.loc['ndvi_mean','temp_mean_c']:.2f}).

**Why it matters:** soil moisture integrates recent rainfall, so it is the better single
predictor — consistent with `forage_dryness_index` weighting both. The weak same-month
rainfall/NDVI link is the lag effect from chart 2, seen numerically.

**What it does not show:** causation, and it is computed across counties with very different
baselines, so much of the correlation is *between-county*, not within-county. A
within-county correlation would be lower.""")


## 7 · What was imputed


In [ ]:
imp_cols = [c for c in PANEL if c.endswith("_imputed")]
fig, ax = plt.subplots(figsize=(8, 3))
if imp_cols:
    counts = PANEL[imp_cols].sum().sort_values()
    ax.barh(range(len(counts)), counts.values, color="#b23b3b")
    ax.set(yticks=range(len(counts)), yticklabels=[c.replace("_imputed", "") for c in counts.index],
           xlabel="Imputed county-months", title="Imputed cells by variable")
else:
    ax.text(.5, .5, "No imputation flags in this panel", ha="center", va="center")
    ax.axis("off")
save(fig, "07_imputation_flags", "Imputed cells by variable",
f"""Counts of cells filled by same-year-month national mean rather than observed.

**Reading it:** {', '.join(f'{c.replace("_imputed","")}: {int(PANEL[c].sum())}' for c in imp_cols) or 'none'}.

**Why it matters:** imputation is an assumption, not a measurement. These counts are
recorded so a reader can judge how much of any result rests on them. In this panel the
affected variable is soil moisture, consistent with `docs/DATA_MODEL_CARD.md`.

**What it does not show:** whether the imputed values are *good*. Filling from the same month
and year is defensible for a slowly-varying variable and indefensible for one that jumps.""")


## 8 · Write every explanation to disk

Each chart gets its own `.md` beside the image, plus one combined
`CHART_EXPLANATIONS.md` that a reader can open first.


In [ ]:
for stem, text in EXPLANATIONS.items():
    (RUN_DIR / f"{stem}.md").write_text(text, encoding="utf-8")

header = f"""# Chart explanations — exploratory data analysis

Generated by `data_pipeline/notebooks/02_eda.ipynb`.
Panel: `master_cleaned_2015_2025.csv` — {len(PANEL):,} county-months, {PANEL.county.nunique()} counties,
{PANEL.date.min():%Y-%m} to {PANEL.date.max():%Y-%m}.

**Read this first.** These charts describe a harmonised panel of satellite and reanalysis
data. They do not show crop loss, forage availability, or model accuracy. The ML target is
a synthesized "NDVI below the county's 25th percentile" label. See `docs/DATA_MODEL_CARD.md`.

---

"""
(RUN_DIR / "CHART_EXPLANATIONS.md").write_text(header + "\n---\n\n".join(EXPLANATIONS.values()), encoding="utf-8")

# A machine-readable index so other tooling can find the figures without globbing
(RUN_DIR / "figures.json").write_text(json.dumps(
    [{"file": f"figures/{s}.png", "explanation": f"{s}.md", "title": EXPLANATIONS[s].splitlines()[0].lstrip('# ')}
     for s in EXPLANATIONS], indent=2), encoding="utf-8")

print(f"Wrote {len(EXPLANATIONS)} charts, {len(EXPLANATIONS)} per-chart explanations,")
print(f"CHART_EXPLANATIONS.md and figures.json to {RUN_DIR.resolve()}")


In [ ]:
summary = pd.DataFrame([
    {"chart": s, "explanation": f"{s}.md"} for s in EXPLANATIONS
])
display(summary)
print(f"\nOne combined document: {RUN_DIR / 'CHART_EXPLANATIONS.md'}")
